# Pareto Incentive Groups with NEAT Proportion Matching

This notebook extends grouped federal incentive optimization by adding a target-distribution objective
from a pre-trained model artifact in `Examples/Proportional_NEAT`.

The artifact is a `Projection` object containing `panel_placements` by ZIP. We convert that to target
proportions and optimize grouped incentive policies so predicted new-installation proportions match it.

In [1]:
from pathlib import Path
import json
import pickle
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.preprocessing import StandardScaler
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False
    print("scikit-learn not available; install scikit-learn to run clustering")

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "Models").exists():
    repo_root = repo_root.parent

if not (repo_root / "Models").exists():
    raise RuntimeError("Could not locate repository root containing 'Models' directory.")

import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Data.data_load_util import make_dataset
from Simulation.projections_util import create_paper_objectives

plt.style.use("seaborn-v0_8")

In [2]:
# --- Configuration ---
N_VALUES = [1, 2, 3, 5, 8]
INCENTIVE_GRID = list(range(-2000, 8001, 500))
YEARS_TO_SIMULATE = 5
TARGET_ADOPTION_MODE = "additional_only"
REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION = 0

# Optimization budgets
WEIGHT_SAMPLES = 100
LOCAL_SEARCH_STEPS = 70
REFINE_POP_SIZE = 100
REFINE_GENERATIONS = 28
REFINE_MUTATION_RATE = 0.25

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

FEDERAL_CACHE_DIR = repo_root / "Examples" / "model_cache" / "federal"
EXPERIMENT_CACHE_DIR = repo_root / "Examples" / "model_cache" / "pareto_neat_match"
RESULTS_DIR = repo_root / "Examples" / "pareto_neat_match_results"
EXPERIMENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NEAT_PROJECTION_PATH = repo_root / "Examples" / "Proportional_NEAT" / "PROJ_softmax_b0p15_large.pkl"

if not FEDERAL_CACHE_DIR.exists():
    raise RuntimeError("Federal payload cache not found. Run federal cache generation first.")
if not NEAT_PROJECTION_PATH.exists():
    raise RuntimeError(f"Missing NEAT projection file: {NEAT_PROJECTION_PATH}")
if not SKLEARN_OK:
    raise RuntimeError("scikit-learn required for this notebook.")

In [3]:
state_behavior_df = pd.read_csv(repo_root / "Models" / "Incentives" / "state_behavior.csv")
zips_df, _, _ = make_dataset(granularity="both", remove_outliers=False, load_dir_prefix=str(repo_root / "Data") + "/")
zip_lookup = zips_df.set_index("region_name")
objectives = create_paper_objectives()

with open(NEAT_PROJECTION_PATH, "rb") as f:
    neat_projection = pickle.load(f)

if not hasattr(neat_projection, "panel_placements"):
    raise RuntimeError("Loaded NEAT artifact does not contain panel_placements")

raw_target = neat_projection.panel_placements

target_panels = {}
for k, v in raw_target.items():
    try:
        zip_int = int(k)
    except Exception:
        continue
    if zip_int in zip_lookup.index:
        target_panels[zip_int] = float(v)

target_total = sum(target_panels.values())
target_prop = {z: (p / target_total if target_total > 0 else 0.0) for z, p in target_panels.items()}
print(f"Loaded NEAT target proportions for {len(target_prop)} ZIPs")
print(f"Total target panels in overlapping ZIPs: {target_total:,.0f}")

Loaded NEAT target proportions for 10421 ZIPs
Total target panels in overlapping ZIPs: 2,000,000


In [5]:
def build_target_adoption_curve(base_adoption, annual_increment, years_total, mode):
    out = []
    for year in range(1, years_total + 1):
        if mode == "absolute_total":
            t = base_adoption + year * annual_increment
        else:
            t = year * annual_increment
        out.append(float(np.clip(t, 0.0, 1.0)))
    return out

def required_incentives_from_payload(payload, cutoff):
    return payload["installation_cost"] - payload["yearly_savings_coeff"] * cutoff

def eval_cutoff_adoption(payloads, cutoff, threshold):
    vals = []
    for payload in payloads:
        req = required_incentives_from_payload(payload, cutoff)
        vals.append(float(np.mean(req <= threshold)))
    return float(np.mean(vals)) if vals else 0.0

def calibrate_cutoffs(payloads, yearly_targets, threshold, lower=0.25, upper=30.0, steps=30):
    cutoffs = []
    prev = lower
    for target in yearly_targets:
        lo, hi = prev, upper
        if eval_cutoff_adoption(payloads, hi, threshold) < target:
            cutoffs.append(hi)
            prev = hi
            continue
        if eval_cutoff_adoption(payloads, lo, threshold) >= target:
            cutoffs.append(lo)
            prev = lo
            continue
        for _ in range(steps):
            mid = 0.5 * (lo + hi)
            if eval_cutoff_adoption(payloads, mid, threshold) < target:
                lo = mid
            else:
                hi = mid
        cutoffs.append(hi)
        prev = hi
    return cutoffs

def load_state_payloads(cache_dir):
    state_payloads = {}
    for p in sorted(cache_dir.glob("zip_payloads_*.pkl")):
        parts = p.stem.split("_")
        if len(parts) < 3:
            continue
        state_code = parts[2]
        with open(p, "rb") as f:
            cached = pickle.load(f)
        payloads = cached.get("payloads", [])
        if payloads:
            if state_code not in state_payloads or len(payloads) > len(state_payloads[state_code]):
                state_payloads[state_code] = payloads
    return state_payloads

state_payloads = load_state_payloads(FEDERAL_CACHE_DIR)
state_cutoffs = {}

for state_code in tqdm(sorted(state_payloads.keys()), desc="Calibrating state cutoffs", unit="state"):
    row = state_behavior_df[state_behavior_df["State code"] == state_code]
    if row.empty:
        continue
    row = row.iloc[0]
    target_curve = build_target_adoption_curve(
        float(row["prop_adopted_status_quo"][1:-1]),
        float(row["prop_adopted_per_year_average"]),
        YEARS_TO_SIMULATE,
        TARGET_ADOPTION_MODE,
    )
    threshold = -REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION
    state_cutoffs[state_code] = calibrate_cutoffs(state_payloads[state_code], target_curve, threshold)

print(f"Cutoffs prepared for {len(state_cutoffs)} states")

Calibrating state cutoffs: 100%|██████████| 50/50 [00:15<00:00,  3.19state/s]

Cutoffs prepared for 50 states


In [6]:
zip_rows = []
for state_code, payloads in state_payloads.items():
    if state_code not in state_cutoffs:
        continue
    final_cutoff = state_cutoffs[state_code][-1]
    for payload in payloads:
        zip_code = int(payload["zip"])
        if zip_code not in zip_lookup.index:
            continue
        z = zip_lookup.loc[zip_code]
        coeff = payload["yearly_savings_coeff"]
        zip_rows.append(
            {
                "zip": zip_code,
                "state_code": state_code,
                "count_qualified": float(z["count_qualified"]),
                "sunlight": float(z["yearly_sunlight_kwh_kw_threshold_avg"]),
                "carbon_per_panel": float(z["carbon_offset_metric_tons_per_panel"]),
                "median_income": float(z["Median_income"]),
                "black_prop": float(z["black_prop"]),
                "panel_utilization": float(z["panel_utilization"]),
                "coeff_mean": float(np.mean(coeff)),
                "coeff_std": float(np.std(coeff)),
                "state_cutoff_final": float(final_cutoff),
                "payload": payload,
                "target_prop": float(target_prop.get(zip_code, 0.0)),
            }
        )

zip_df = pd.DataFrame(zip_rows)
print(f"Optimization ZIP universe: {len(zip_df)}")
print(f"ZIPs with nonzero target proportion: {(zip_df['target_prop'] > 0).sum()}")

Optimization ZIP universe: 10475
ZIPs with nonzero target proportion: 10352


In [7]:
def cluster_for_n(df, n_groups, cache_dir):
    cp = cache_dir / f"clusters_n{n_groups}.csv"
    if cp.exists():
        return pd.read_csv(cp)
    features = [
        "sunlight", "carbon_per_panel", "median_income", "black_prop",
        "panel_utilization", "coeff_mean", "coeff_std", "state_cutoff_final", "target_prop"
    ]
    X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0.0).values
    Xs = StandardScaler().fit_transform(X)
    km = MiniBatchKMeans(n_clusters=n_groups, random_state=RANDOM_SEED, n_init=10, batch_size=1024)
    labels = km.fit_predict(Xs)
    out = df[["zip", "state_code"]].copy()
    out["group_id"] = labels.astype(int)
    out.to_csv(cp, index=False)
    return out

def compute_metrics(placements):
    vals = {}
    for obj in objectives:
        vals[obj.name] = float(obj.calc(zips_df, placements))
    return vals

def match_metrics(predicted_panels, target_prop_map):
    total_pred = sum(predicted_panels.values())
    if total_pred <= 0:
        return {"Match_MAE": 1.0, "Match_RMSE": 1.0, "Match_Score": 0.0}

    pred_prop = {z: p / total_pred for z, p in predicted_panels.items()}
    keys = sorted(set(pred_prop.keys()) | set(target_prop_map.keys()))
    diffs = np.array([pred_prop.get(k, 0.0) - target_prop_map.get(k, 0.0) for k in keys], dtype=float)
    mae = float(np.mean(np.abs(diffs)))
    rmse = float(np.sqrt(np.mean(diffs * diffs)))
    # score in [0,1], higher better
    score = float(max(0.0, 1.0 - mae * len(keys) / 2.0))
    return {"Match_MAE": mae, "Match_RMSE": rmse, "Match_Score": score}

def evaluate_strategy(df, cluster_df, incentives_by_group):
    merged = df[["zip", "state_code", "count_qualified", "payload", "target_prop"]].merge(cluster_df, on=["zip", "state_code"], how="inner")
    placements = {}
    total_panels = 0.0
    total_spending = 0.0
    target_map = {}

    for row in merged.itertuples(index=False):
        incentive = incentives_by_group[int(row.group_id)]
        threshold = -incentive
        cutoff = state_cutoffs[row.state_code][-1]
        required = required_incentives_from_payload(row.payload, cutoff)
        adoption = float(np.mean(required <= threshold))
        panels = max(0.0, adoption * float(row.count_qualified))
        placements[int(row.zip)] = panels
        target_map[int(row.zip)] = float(row.target_prop)
        total_panels += panels
        total_spending += incentive * panels

    metric_vals = compute_metrics(placements)
    mm = match_metrics(placements, target_map)
    return {
        "Carbon Offset": metric_vals.get("Carbon Offset", np.nan),
        "Energy Generation": metric_vals.get("Energy Potential", np.nan),
        "Racial Equity": metric_vals.get("Racial Equity", np.nan),
        "Income Equity": metric_vals.get("Income Equity", np.nan),
        "Panels": total_panels,
        "Spending": total_spending,
        "Match_MAE": mm["Match_MAE"],
        "Match_RMSE": mm["Match_RMSE"],
        "Match_Score": mm["Match_Score"],
        "placements": placements,
    }

def dominates(a, b):
    max_keys = ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity", "Panels", "Match_Score"]
    better_or_equal = all(b[k] >= a[k] for k in max_keys) and b["Spending"] <= a["Spending"]
    strict = any(b[k] > a[k] for k in max_keys) or b["Spending"] < a["Spending"]
    return better_or_equal and strict

def pareto_filter(rows):
    keep = []
    for i, r in enumerate(rows):
        dom = False
        for j, o in enumerate(rows):
            if i == j:
                continue
            if dominates(r, o):
                dom = True
                break
        if not dom:
            keep.append(r)
    return keep

In [8]:
def random_strategy(n_groups):
    return tuple(int(np.random.choice(INCENTIVE_GRID)) for _ in range(n_groups))

def neighbors(strategy):
    out = []
    for i in range(len(strategy)):
        idx = INCENTIVE_GRID.index(strategy[i])
        for d in (-1, 1):
            j = idx + d
            if 0 <= j < len(INCENTIVE_GRID):
                s = list(strategy)
                s[i] = INCENTIVE_GRID[j]
                out.append(tuple(s))
    return out

def normalize_and_score(rows, weights):
    cols = ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity", "Panels", "Match_Score", "Spending"]
    mat = pd.DataFrame([{c: r[c] for c in cols} for r in rows])
    mins, maxs = mat.min(), mat.max()
    def score(r):
        s = 0.0
        for c in ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity", "Panels", "Match_Score"]:
            den = max(maxs[c] - mins[c], 1e-9)
            s += weights[c] * (r[c] - mins[c]) / den
        den_s = max(maxs["Spending"] - mins["Spending"], 1e-9)
        s += weights["Spending"] * (maxs["Spending"] - r["Spending"]) / den_s
        return s
    return score

def weighted_stage(df, cluster_df, n_groups):
    eval_cache = {}
    pool = []

    seeds = [random_strategy(n_groups) for _ in range(140)]
    for s in tqdm(seeds, desc=f"Seed eval n={n_groups}", leave=False):
        if s not in eval_cache:
            eval_cache[s] = evaluate_strategy(df, cluster_df, s)
        pool.append({"strategy": s, **eval_cache[s]})

    for _ in range(WEIGHT_SAMPLES):
        wv = np.random.dirichlet(np.ones(7))
        weights = {
            "Carbon Offset": wv[0], "Energy Generation": wv[1], "Racial Equity": wv[2],
            "Income Equity": wv[3], "Panels": wv[4], "Match_Score": wv[5], "Spending": wv[6]
        }
        scorer = normalize_and_score(pool, weights)
        cur = random_strategy(n_groups)
        if cur not in eval_cache:
            eval_cache[cur] = evaluate_strategy(df, cluster_df, cur)
        best, best_s = cur, scorer(eval_cache[cur])

        for _ in range(LOCAL_SEARCH_STEPS):
            improved = False
            cand_list = neighbors(best)
            random.shuffle(cand_list)
            for cand in cand_list[:6]:
                if cand not in eval_cache:
                    eval_cache[cand] = evaluate_strategy(df, cluster_df, cand)
                sc = scorer(eval_cache[cand])
                if sc > best_s:
                    best, best_s = cand, sc
                    improved = True
            if not improved:
                break
        pool.append({"strategy": best, **eval_cache[best]})

    uniq = {r["strategy"]: r for r in pool}
    return pareto_filter(list(uniq.values())), eval_cache

def mutate(strategy):
    s = list(strategy)
    for i in range(len(s)):
        if random.random() < REFINE_MUTATION_RATE:
            idx = INCENTIVE_GRID.index(s[i])
            idx2 = min(max(idx + random.choice([-1, 1]), 0), len(INCENTIVE_GRID) - 1)
            s[i] = INCENTIVE_GRID[idx2]
    return tuple(s)

def refine_stage(df, cluster_df, n_groups, seed_front, eval_cache):
    pop = [r["strategy"] for r in seed_front[:REFINE_POP_SIZE]]
    while len(pop) < REFINE_POP_SIZE:
        pop.append(random_strategy(n_groups))

    for _ in tqdm(range(REFINE_GENERATIONS), desc=f"Refine n={n_groups}", leave=False):
        cand = pop + [mutate(s) for s in pop]
        eval_rows = []
        for s in cand:
            if s not in eval_cache:
                eval_cache[s] = evaluate_strategy(df, cluster_df, s)
            eval_rows.append({"strategy": s, **eval_cache[s]})
        front = pareto_filter(eval_rows)
        random.shuffle(front)
        pop = [r["strategy"] for r in front[:REFINE_POP_SIZE]]
        while len(pop) < REFINE_POP_SIZE:
            pop.append(random_strategy(n_groups))

    final_rows = []
    for s in set(pop):
        if s not in eval_cache:
            eval_cache[s] = evaluate_strategy(df, cluster_df, s)
        final_rows.append({"strategy": s, **eval_cache[s]})
    return pareto_filter(final_rows)

def best_match_local_search(df, cluster_df, n_groups):
    best = random_strategy(n_groups)
    best_eval = evaluate_strategy(df, cluster_df, best)
    for _ in range(220):
        improved = False
        for cand in neighbors(best):
            ev = evaluate_strategy(df, cluster_df, cand)
            # maximize match, tie-break by lower spending
            if (ev["Match_Score"] > best_eval["Match_Score"]) or (
                ev["Match_Score"] == best_eval["Match_Score"] and ev["Spending"] < best_eval["Spending"]
            ):
                best, best_eval = cand, ev
                improved = True
        if not improved:
            break
    return {"strategy": best, **best_eval}

In [9]:
all_rows = []
best_match_rows = []
runtime_rows = []

for n in tqdm(N_VALUES, desc="Optimize across n", unit="n"):
    t0 = time.time()
    cdf = cluster_for_n(zip_df, n, EXPERIMENT_CACHE_DIR)

    seed_front, cache_eval = weighted_stage(zip_df, cdf, n)
    refined_front = refine_stage(zip_df, cdf, n, seed_front, cache_eval)
    final_front = pareto_filter(seed_front + refined_front)
    best_match = max(final_front, key=lambda r: (r["Match_Score"], -r["Spending"])) if final_front else best_match_local_search(zip_df, cdf, n)

    # Dedicated match objective search
    best_match_only = best_match_local_search(zip_df, cdf, n)

    for r in final_front:
        all_rows.append(
            {
                "n_groups": n,
                "strategy": json.dumps(list(r["strategy"])),
                "Carbon Offset": r["Carbon Offset"],
                "Energy Generation": r["Energy Generation"],
                "Racial Equity": r["Racial Equity"],
                "Income Equity": r["Income Equity"],
                "Panels": r["Panels"],
                "Spending": r["Spending"],
                "Match_Score": r["Match_Score"],
                "Match_MAE": r["Match_MAE"],
                "Match_RMSE": r["Match_RMSE"],
            }
        )

    best_match_rows.append(
        {
            "n_groups": n,
            "best_from_pareto_strategy": json.dumps(list(best_match["strategy"])),
            "best_from_pareto_match": best_match["Match_Score"],
            "match_only_strategy": json.dumps(list(best_match_only["strategy"])),
            "match_only_score": best_match_only["Match_Score"],
            "match_only_spending": best_match_only["Spending"],
        }
    )

    runtime_rows.append({"n_groups": n, "elapsed_seconds": time.time() - t0, "pareto_points": len(final_front)})

pareto_df = pd.DataFrame(all_rows)
best_match_df = pd.DataFrame(best_match_rows)
runtime_df = pd.DataFrame(runtime_rows)

pareto_df.to_csv(RESULTS_DIR / "pareto_neat_match_front_all_n.csv", index=False)
best_match_df.to_csv(RESULTS_DIR / "best_match_strategies_by_n.csv", index=False)
runtime_df.to_csv(RESULTS_DIR / "runtime_summary.csv", index=False)

display(runtime_df)
display(best_match_df)
display(pareto_df.head())

Optimize across n:  20%|██        | 1/5 [1:06:22<4:25:31, 3982.89s/n]


KeyboardInterrupt: 

In [ ]:
if pareto_df.empty:
    raise RuntimeError("No Pareto results generated. Increase search budgets.")

# 1) Match quality vs n
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(best_match_df["n_groups"], best_match_df["best_from_pareto_match"], marker="o", label="Best match on Pareto front")
ax.plot(best_match_df["n_groups"], best_match_df["match_only_score"], marker="s", label="Dedicated match-only optimization")
ax.set_title("NEAT Proportion Match Quality vs Number of Groups")
ax.set_xlabel("n groups")
ax.set_ylabel("Match score (higher better)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# 2) Spending vs match score by n
fig, ax = plt.subplots(figsize=(7, 4))
for n, g in pareto_df.groupby("n_groups"):
    ax.scatter(g["Spending"], g["Match_Score"], label=f"n={n}", alpha=0.75)
ax.set_title("Spending vs NEAT Match Score (Pareto set)")
ax.set_xlabel("Federal spending")
ax.set_ylabel("Match score")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# 3) Core objectives vs match score
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
obj_cols = ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity"]
for ax, col in zip(axes.flatten(), obj_cols):
    for n, g in pareto_df.groupby("n_groups"):
        ax.scatter(g["Match_Score"], g[col], label=f"n={n}", alpha=0.65)
    ax.set_title(f"{col} vs Match Score")
    ax.set_xlabel("Match score")
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=min(len(N_VALUES), 6))
plt.tight_layout()
plt.show()